# 2차 — 새 후보 2개 생성 (Keep-NaN LightGBM + MLP)

## 배경
전기수 1등 노트북(`ver60_strict_ecdf...`) 분석 결과: 개별 전처리/피처 변경은 단일 모델에
노이즈 수준 영향만 있었음(우리 결론과 동일). 그쪽의 진짜 강점은 **여러 약한 변형을 많이
쌍아 올린 멀티레벨 스태킹**. 우리도 같은 원리로, 새로운 후보 2개를 만들어서
`2차_greedy_ensemble_selection.ipynb`의 풀에 추가합니다.

## 후보 1: LightGBM + Keep-NaN 전처리
그 노트북의 핵심 아이디어를 반영(train 통계만 사용, 규칙 준수):
- DI 구조적 결측을 `-1`로 안 채우고 진짜 NaN 유지 + `_missing`/`_not_applicable_DI` 플래그
- 배아/난자 개수 컬럼 아웃라이어 클리핑 + `_high_flag`
- `배아 이식 경과일` 임계값 플래그(0~1일/3일/5일+)
- `safe_ratio`: 분모가 0/결측이면 진짜 NaN + `_available` 플래그 (저희가 전에 쓴 `/(x+1)`보다 정확)
- `effective_maternal_age_mid`: 기증 난자면 환자 나이 대신 기증자 나이로 교체한 "실효 나이"
- 네이티브 category dtype (LabelEncoder 대신)

## 후보 2: MLP (완전히 다른 모델 타입)
지금까지 시도한 모든 모델(LightGBM/XGBoost/CatBoost/TabTransformer/EBM)이 부스팅 계열이거나
EBM도 결국 0.987 상관관계였음. sklearn MLPClassifier는 구조적으로 완전히 다름 — 디커럴레이션
가능성을 한 번 더 보는 것.

## 누수 방지
- 카테고리/스케일러는 모두 train에만 fit
- ratio의 NaN/0 처리는 행 단위 deterministic (test 통계 미사용)

---
**저장 위치:** `experiment_history/2차/2차_new_candidates.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
BLEND_DIR = SHARED_DIR / "blend_cache"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

N_SPLITS = 5
SEED = 1004

EMBRYO_COUNT_COLS = ["총 생성 배아 수","미세주입된 난자 수","미세주입에서 생성된 배아 수","이식된 배아 수",
    "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수","해동된 배아 수",
    "해동 난자 수","수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수"]
EMBRYO_BINARY_COLS = ["단일 배아 이식 여부","착상 전 유전 진단 사용 여부","동결 배아 사용 여부",
    "신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
CLIP_RULES = {"총 생성 배아 수":40,"수집된 신선 난자 수":40,"미세주입된 난자 수":45,
    "혼합된 난자 수":40,"저장된 배아 수":30,"배아 이식 경과일":7,"난자 혼합 경과일":7,
    "배아 해동 경과일":2,"난자 해동 경과일":1}
RATIO_SPECS = [
    ("총 생성 배아 수", "혼합된 난자 수", "fertilization_rate"),
    ("미세주입에서 생성된 배아 수", "미세주입된 난자 수", "icsi_fertilization_rate"),
    ("이식된 배아 수", "총 생성 배아 수", "embryo_utilization_rate"),
    ("저장된 배아 수", "총 생성 배아 수", "embryo_freezing_rate"),
    ("혼합된 난자 수", "수집된 신선 난자 수", "oocyte_utilization_rate"),
    ("이식된 배아 수", "해동된 배아 수", "thawed_embryo_transfer_ratio"),
]


## 1. 데이터 로드 + 공통 베이스 전처리

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]

y = train_raw[TARGET_COL].values.astype(int)


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def safe_ratio(df, numerator, denominator, new_col):
    df = df.copy()
    can = df[numerator].notna() & df[denominator].notna() & (df[denominator] > 0)
    df[f"{new_col}_available"] = can.astype(int)
    df[new_col] = np.where(can, df[numerator] / df[denominator], np.nan)
    return df


def build_keepnan_features(df):
    df = base_features(df).drop(columns=[c for c in DEAD_COLS if c in df.columns])

    embryo_cols_present = [c for c in (EMBRYO_COUNT_COLS + EMBRYO_BINARY_COLS) if c in df.columns]
    for col in embryo_cols_present:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        is_di_missing = (df["시술 유형"] == "DI") & df[col].isna()
        df[f"{col}_not_applicable_DI"] = is_di_missing.astype(int)
        # NaN은 그대로 유지 (의도적으로 안 채움)

    for col, upper in CLIP_RULES.items():
        if col in df.columns:
            df[f"{col}_high_flag"] = (df[col] > upper).astype(int)
            df[col] = df[col].clip(upper=upper)

    if "배아 이식 경과일" in df.columns:
        df["transfer_day_0_1_flag"] = df["배아 이식 경과일"].isin([0, 1]).astype(int)
        df["transfer_day_3_flag"] = (df["배아 이식 경과일"] == 3).astype(int)
        df["transfer_day_5_or_more_flag"] = (df["배아 이식 경과일"] >= 5).astype(int)

    for num, den, new in RATIO_SPECS:
        if num in df.columns and den in df.columns:
            df = safe_ratio(df, num, den, new)

    # 실효 나이: 기증 난자면 환자 나이 대신 기증자 나이 중간값 사용
    patient_mid = {"만18-34세":31,"만35-37세":36,"만38-39세":38.5,"만40-42세":41,
                    "만43-44세":43.5,"만45-50세":47.5,"알 수 없음":np.nan}
    donor_mid = {"만20세 이하":20,"만21-25세":23,"만26-30세":28,"만31-35세":33,
                 "만36-40세":38,"만41-45세":43,"알 수 없음":np.nan}
    if "난자 출처" in df.columns and "시술 당시 나이" in df.columns:
        df["patient_age_mid"] = df["시술 당시 나이"].map(patient_mid)
        donor_age_mid = df["난자 기증자 나이"].map(donor_mid) if "난자 기증자 나이" in df.columns else np.nan
        donor_known = (df["난자 출처"] == "기증 제공") & donor_age_mid.notna()
        df["effective_maternal_age_mid"] = df["patient_age_mid"]
        df.loc[donor_known, "effective_maternal_age_mid"] = donor_age_mid[donor_known]
        df["donor_rejuvenation_gap"] = 0.0
        df.loc[donor_known, "donor_rejuvenation_gap"] = (
            df.loc[donor_known, "patient_age_mid"] - donor_age_mid[donor_known]
        )

    return df


train_feat = build_keepnan_features(train_raw.drop(columns=["ID"]))
test_feat = build_keepnan_features(test_raw_full.drop(columns=["ID"]))

X_train_kn = train_feat.drop(columns=[TARGET_COL])
X_test_kn = test_feat.copy()

# ★ 대회 규칙 준수: 카테고리는 train 기준으로만 정의, test는 transform만
obj_cols = X_train_kn.select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    train_categories = sorted(X_train_kn[col].dropna().astype(str).unique())
    X_train_kn[col] = pd.Categorical(X_train_kn[col].astype(str), categories=train_categories)
    X_test_kn[col] = pd.Categorical(X_test_kn[col].astype(str), categories=train_categories)

X_test_kn = X_test_kn[X_train_kn.columns]
cat_cols_kn = obj_cols

print(f"Keep-NaN 피처셋: train {X_train_kn.shape}, test {X_test_kn.shape}, 범주형 {len(cat_cols_kn)}개")
print(f"NaN 비율: {X_train_kn.isna().mean().mean():.2%}")


Keep-NaN 피처셋: train (256351, 134), test (90067, 134), 범주형 20개
NaN 비율: 6.67%


## 2. 후보 1: LightGBM + Keep-NaN (5-fold)

In [4]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_kn = np.zeros(len(y))
test_kn = np.zeros(len(X_test_kn))

params_kn = dict(n_estimators=2000, learning_rate=0.02, num_leaves=127, max_depth=6,
                  min_child_samples=50, random_state=SEED, verbose=-1, n_jobs=-1)

start = time.time()
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_kn, y)):
    m = LGBMClassifier(**params_kn)
    m.fit(X_train_kn.iloc[tr_idx], y[tr_idx])
    oof_kn[val_idx] = m.predict_proba(X_train_kn.iloc[val_idx])[:, 1]
    test_kn += m.predict_proba(X_test_kn)[:, 1] / N_SPLITS
    print(f"[fold {fold}] AUC={roc_auc_score(y[val_idx], oof_kn[val_idx]):.5f}  ({time.time()-start:.0f}s 누적)")

print(f"\nLightGBM Keep-NaN OOF AUC: {roc_auc_score(y, oof_kn):.5f}")

BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "oof_lgbm_keepnan.npy", oof_kn)
np.save(BLEND_DIR / "test_lgbm_keepnan.npy", test_kn)
print("저장 완료: oof_lgbm_keepnan.npy, test_lgbm_keepnan.npy")


[fold 0] AUC=0.73994  (18s 누적)
[fold 1] AUC=0.73695  (35s 누적)
[fold 2] AUC=0.73434  (51s 누적)
[fold 3] AUC=0.73527  (67s 누적)
[fold 4] AUC=0.73613  (83s 누적)

LightGBM Keep-NaN OOF AUC: 0.73648
저장 완료: oof_lgbm_keepnan.npy, test_lgbm_keepnan.npy


## 3. 후보 2: MLP (완전히 다른 모델 타입, 표준 인코딩 + 스케일링)

In [5]:
def base_features2(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df

def fill_na2(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df

train_mlp = fill_na2(base_features2(train_raw.drop(columns=["ID"]).copy()).drop(columns=DEAD_COLS))
test_mlp_df = fill_na2(base_features2(test_raw_full.drop(columns=["ID"]).copy()).drop(columns=DEAD_COLS))
cat_cols_mlp = train_mlp.select_dtypes(include=["object"]).columns.tolist()

X_train_mlp = train_mlp.drop(columns=[TARGET_COL]).copy()
X_test_mlp = test_mlp_df.copy()

# ★ 대회 규칙 준수: LabelEncoder는 train에만 fit
for col in cat_cols_mlp:
    le = LabelEncoder()
    train_vals = X_train_mlp[col].astype(str)
    le.fit(train_vals)
    known = set(le.classes_)
    fallback = train_vals.value_counts().idxmax()
    test_vals = X_test_mlp[col].astype(str)
    test_vals = test_vals.where(test_vals.isin(known), fallback)
    X_train_mlp[col] = le.transform(train_vals)
    X_test_mlp[col] = le.transform(test_vals)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_mlp = np.zeros(len(y))
test_mlp_pred = np.zeros(len(X_test_mlp))

start = time.time()
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_mlp, y)):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train_mlp.iloc[tr_idx])
    X_val_s = scaler.transform(X_train_mlp.iloc[val_idx])
    X_test_s = scaler.transform(X_test_mlp)

    m = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=1e-3,
        early_stopping=True,
        n_iter_no_change=15,
        max_iter=300,
        random_state=SEED,
    )
    m.fit(X_tr_s, y[tr_idx])
    oof_mlp[val_idx] = m.predict_proba(X_val_s)[:, 1]
    test_mlp_pred += m.predict_proba(X_test_s)[:, 1] / N_SPLITS
    print(f"[fold {fold}] AUC={roc_auc_score(y[val_idx], oof_mlp[val_idx]):.5f}  ({time.time()-start:.0f}s 누적)")

print(f"\nMLP OOF AUC: {roc_auc_score(y, oof_mlp):.5f}")

np.save(BLEND_DIR / "oof_mlp.npy", oof_mlp)
np.save(BLEND_DIR / "test_mlp.npy", test_mlp_pred)
print("저장 완료: oof_mlp.npy, test_mlp.npy")


[fold 0] AUC=0.73684  (8s 누적)
[fold 1] AUC=0.73277  (20s 누적)
[fold 2] AUC=0.72992  (32s 누적)
[fold 3] AUC=0.73237  (39s 누적)
[fold 4] AUC=0.73592  (45s 누적)

MLP OOF AUC: 0.73313
저장 완료: oof_mlp.npy, test_mlp.npy


## 4. 기존 baseline과 상관관계 확인

In [6]:
LGBM_OOF_PATH = BLEND_DIR / "oof_10seed_bagged.npy"
if LGBM_OOF_PATH.exists():
    oof_baseline = np.load(LGBM_OOF_PATH)
    print(f"baseline(10시드 배깅) AUC: {roc_auc_score(y, oof_baseline):.5f}")
    print(f"lgbm_keepnan 상관관계: {np.corrcoef(oof_baseline, oof_kn)[0,1]:.4f}")
    print(f"mlp 상관관계:          {np.corrcoef(oof_baseline, oof_mlp)[0,1]:.4f}")
    print("(참고: 지금까지 본 모델 다양화 시도는 전부 0.96+ 였음. EBM도 0.987)")
else:
    print(f"{LGBM_OOF_PATH} 없음 — 직접 비교해보세요")


baseline(10시드 배깅) AUC: 0.74036
lgbm_keepnan 상관관계: 0.9764
mlp 상관관계:          0.9644
(참고: 지금까지 본 모델 다양화 시도는 전부 0.96+ 였음. EBM도 0.987)


## 5. 다음 단계

이 두 후보(`oof_lgbm_keepnan`/`test_lgbm_keepnan`, `oof_mlp`/`test_mlp`)가 `blend_cache`에
저장됐으니, **`2차_greedy_ensemble_selection.ipynb`를 다시 실행**하면 자동으로 풀에 포함됩니다.
상관관계가 의미있게 낮다면(특히 MLP) greedy가 채택할 가능성이 있고, 그러면 0.74071을 넘는
첫 사례가 될 수 있습니다.
